# 03 深度分析:时间分布、转化漏斗与用户分群

本 Notebook 在 01 建库、02 概览的基础上,做三类深度分析:

1. 小时维度:24 小时活跃用户与浏览深度分布
2. 转化漏斗:用户级 PV → 加购/收藏 → 购买 的转化路径
3. 用户分群:购买频次分层,识别一次性买家与复购用户

口径说明:
- 所有 `_records` 是行为次数,不是订单数。
- 本数据为公开脱敏行为流(非全量曝光),购买转化率会显著高于真实电商场景,
  结论以"横向对比/相对趋势"为主,不直接等同于业务绝对值。

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import display
import matplotlib.pyplot as plt

from src.config import DB_FILE, FIGURE_DIR
from src.db import get_con
from src.plotting import setup_chinese_font

print("当前工作目录:", Path.cwd())
print("数据库:", DB_FILE)
print("图输出:", FIGURE_DIR)

setup_chinese_font()

con = get_con(read_only=True)

In [ ]:
con = get_con(read_only=True)

hourly_activity = con.execute("""
    SELECT
        behavior_hour,
        COUNT(*) AS total_records,
        COUNT(*) FILTER (WHERE behavior_type = 'pv')  AS pv_records,
        COUNT(DISTINCT user_id) AS active_users,
        COUNT(DISTINCT user_id) FILTER (WHERE behavior_type = 'pv')  AS pv_users,
        COUNT(*) FILTER (WHERE behavior_type = 'buy') AS buy_records,
        COUNT(DISTINCT user_id) FILTER (WHERE behavior_type = 'buy') AS buy_users
    FROM user_behavior
    GROUP BY behavior_hour
    ORDER BY behavior_hour
""").df()

hourly_activity["pv_per_active_user"] = (
    hourly_activity["pv_records"] / hourly_activity["active_users"]
).round(2)

display(hourly_activity)

In [ ]:
print("小时数：", len(hourly_activity))
print(
    "小时范围：",
    hourly_activity["behavior_hour"].min(),
    "至",
    hourly_activity["behavior_hour"].max()
)

totals = con.execute("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(*) FILTER (WHERE behavior_type = 'pv') AS pv_records
    FROM user_behavior
""").df().iloc[0]

print("小时 PV 合计:", f"{hourly_activity['pv_records'].sum():,}")
print("小时总记录合计:", f"{hourly_activity['total_records'].sum():,}")
print("库内 PV:", f"{int(totals['pv_records']):,}")
print("库内总记录:", f"{int(totals['total_records']):,}")

assert hourly_activity["pv_records"].sum() == int(totals["pv_records"]), "PV 合计不一致"
assert hourly_activity["total_records"].sum() == int(totals["total_records"]), "总记录不一致"
print("✅ 小时汇总与全表一致。")

In [ ]:
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.bar(
    hourly_activity["behavior_hour"],
    hourly_activity["active_users"],
    color="#7FB3D5", alpha=0.85, label="活跃用户数",
)
ax1.set_xlabel("小时(北京时间)")
ax1.set_ylabel("活跃用户数", color="#2E86C1")
ax1.set_xticks(range(24))
ax1.tick_params(axis="y", labelcolor="#2E86C1")

ax2 = ax1.twinx()
ax2.plot(
    hourly_activity["behavior_hour"],
    hourly_activity["pv_per_active_user"],
    color="#E74C3C", marker="o", linewidth=2, label="人均 PV",
)
ax2.set_ylabel("人均 PV", color="#E74C3C")
ax2.tick_params(axis="y", labelcolor="#E74C3C")

ax1.set_title("24 小时活跃用户与人均浏览深度(2017-11-25 ~ 12-03)")
fig.tight_layout()

out_path = FIGURE_DIR / "hourly_activity.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()

print("图已保存:", out_path)

#小时维度

1. 活跃用户呈"双峰":晚间 21 点最高(68.3 万),20–22 点是主峰;午间 10–13 点(57–59 万)为次峰;凌晨 3–5 点低谷(6–8 万),符合作息规律。

2. 人均浏览深度也在晚间达峰:21 点 11.04、22 点 11.46,显著高于白天(10–18 点约 7.2–8.0)。说明晚间不仅"人多",单人浏览也更深。

3. 购买记录峰值与活跃峰值同步:21 点购买记录 145,431 次为全天最高,与活跃用户峰值重合,晚间是转化黄金时段。

4. 业务建议:大促、push、推荐位投放应集中在 19–22 点;凌晨 0–6 点流量极低,可考虑用于后台任务/资源调度。

转化漏斗:浏览 → 加购/收藏 → 购买

分析问题:
1. 有多少用户走完了"浏览 → 加购/收藏 → 购买"的完整路径?
2. 每一步的转化率是多少?流失最大的一步在哪?
3. 有没有用户跳过前置行为直接购买?

指标定义(均为用户级,不是行为记录级):
- 浏览用户:窗口内至少 1 次 pv 的去重用户
- 意图用户:窗口内至少 1 次 cart 或 fav 的去重用户
- 购买用户:窗口内至少 1 次 buy 的去重用户
- 转化率 = 下一阶段用户数 / 上一阶段用户数



In [ ]:
# 用户级漏斗:每用户打 flag,再汇总


funnel_raw = con.execute("""
    WITH user_flags AS (
        SELECT
            user_id,
            MAX(CASE WHEN behavior_type = 'pv' THEN 1 ELSE 0 END) AS has_pv,
            MAX(CASE WHEN behavior_type IN ('cart','fav') THEN 1 ELSE 0 END) AS has_intent,
            MAX(CASE WHEN behavior_type = 'buy' THEN 1 ELSE 0 END) AS has_buy
        FROM user_behavior
        GROUP BY user_id
    )
    SELECT
        COUNT(*) AS total_users,
        SUM(has_pv) AS pv_users,
        SUM(has_intent) AS intent_users,
        SUM(has_buy) AS buy_users,

        -- 只浏览、无任何意图/购买
        SUM(CASE WHEN has_pv = 1 AND has_intent = 0 AND has_buy = 0 THEN 1 ELSE 0 END)
            AS pv_only_users,

        -- 完整路径:浏览 + 意图 + 购买
        SUM(CASE WHEN has_pv = 1 AND has_intent = 1 AND has_buy = 1 THEN 1 ELSE 0 END)
            AS full_path_users,

        -- 有购买但无前置意图(直接买 / 意图发生在窗口外)
        SUM(CASE WHEN has_buy = 1 AND has_intent = 0 THEN 1 ELSE 0 END)
            AS buy_without_intent_users
    FROM user_flags
""").df().iloc[0]

funnel_raw

In [ ]:
# 转化率 + 合理性校验
import pandas as pd

pv_u     = int(funnel_raw["pv_users"])
intent_u = int(funnel_raw["intent_users"])
buy_u    = int(funnel_raw["buy_users"])

funnel_table = pd.DataFrame({
    "阶段": ["浏览(pv)", "加购/收藏(cart/fav)", "购买(buy)"],
    "用户数": [pv_u, intent_u, buy_u],
})
funnel_table["占浏览用户%"] = (funnel_table["用户数"] / pv_u * 100).round(2)
funnel_table["环比上一步%"] = [
    100.00,
    round(intent_u / pv_u * 100, 2),
    round(buy_u / intent_u * 100, 2),
]

display(funnel_table)

# 校验:漏斗阶段应单调递减(用户级)
assert pv_u >= intent_u >= buy_u, "漏斗不单调,检查口径"
print("✅ 漏斗阶段单调递减。")
print(f"只浏览不买:{int(funnel_raw['pv_only_users']):,}")
print(f"完整路径:{int(funnel_raw['full_path_users']):,}")
print(f"有购买但无前置意图:{int(funnel_raw['buy_without_intent_users']):,}")

In [ ]:
# 漏斗可视化(横向条形 + 转化率标注)
FIGURES = FIGURE_DIR 
FIGURES.mkdir(parents=True, exist_ok=True)

stages = funnel_table["阶段"].tolist()
values = funnel_table["用户数"].tolist()
colors = ["#5DADE2", "#F5B041", "#E74C3C"]

fig, ax = plt.subplots(figsize=(10, 4.5))

y_pos = list(range(len(stages)))[::-1]  # 让第一行在最上面
ax.barh(y_pos, values, color=colors, height=0.55)

base = values[0]
for yi, v, pct in zip(y_pos, values, funnel_table["占浏览用户%"]):
    ax.text(v, yi, f"  {v:,}({pct:.1f}%)", va="center", fontsize=10)

ax.set_yticks(y_pos)
ax.set_yticklabels(stages)
ax.set_xlabel("用户数")
ax.set_xlim(0, base * 1.22)  # 右侧留白给标注
ax.set_title("用户级转化漏斗:浏览 → 加购/收藏 → 购买")

fig.tight_layout()
out_path = FIGURE_DIR / "conversion_funnel.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()

print("图已保存:", out_path)

## 转化漏斗核心发现

1. 整体转化:浏览用户约 XXX 万,最终购买约 YYY 万,累计转化 68.3%。
   加购/收藏环节环比转化 87.3%,购买环节环比 78.3%。

2. 瓶颈在 “加购/收藏 → 购买”:购买转化流失21.7 > 加购/收藏流失的12.7%
    业务侧应重点关注  结算链路、促单手段(如购物车降价提醒)。

3. 只浏览不买用户约 31.1 万,占浏览用户 31.7%,是最大的待激活群体;
   对应业务动作:针对"浏览多、无加购/收藏"的用户做定向优惠券或推荐。

4. 存在显著的高效“静默下单”群体:有 69,141 人(约 10.3%) 在无前置加购/收藏的情况下直接完成购买。
   这可能是因为:
   - 用户属于“目标明确型”,搜完即买;
   - 或者其加购/收藏行为发生在本数据集的 9 天窗口期之外(历史意图在窗口外转化)。
   针对这类短链路、高意向用户,建议在搜索结果页和详情页直接提供“一键购买”或“限时秒杀”,缩短决策链路。

#用户分群:购买频次分层

分析问题:
1. 购买用户中有多少人只买了一次?多少人买两次及以上?
2. 高频买家贡献了多少购买行为?
3. 购买频次呈长尾还是二八分布?

指标定义:
- 按 `user_id` 聚合 `buy` 行为的记录数作为“购买频次”。
- 频次 1:一次性买家(one-time buyer)
- 频次 2:复购买家
- 频次 3+:高频买家

口径提示:由于本数据集仅包含 9 天窗口,频次仅代表“窗口内购买次数”,
并非真实的长期复购周期。

In [ ]:
#购买频次分层查询
freq_df = con.execute("""
    WITH user_buy_count AS (
        SELECT
            user_id,
            COUNT(*) AS buy_count
        FROM user_behavior
        WHERE behavior_type = 'buy'
        GROUP BY user_id
    )
    SELECT
        CASE
            WHEN buy_count = 1 THEN '1次(一次性)'
            WHEN buy_count = 2 THEN '2次(复购)'
            ELSE '3次及以上(高频)'
        END AS freq_group,
        COUNT(*) AS user_count,
        SUM(buy_count) AS total_buy_records
    FROM user_buy_count
    GROUP BY freq_group
""").df()

freq_df["order"] = freq_df["freq_group"].map({"1次(一次性)": 1, "2次(复购)": 2, "3次及以上(高频)": 3})
freq_df = freq_df.sort_values("order").drop(columns="order").reset_index(drop=True)

# 计算占比
total_buy_users = freq_df["user_count"].sum()
total_buy_records = freq_df["total_buy_records"].sum()

freq_df["用户占比%"] = (freq_df["user_count"] / total_buy_users * 100).round(2)
freq_df["购买贡献%"] = (freq_df["total_buy_records"] / total_buy_records * 100).round(2)

display(freq_df)

In [ ]:
#购买频次分层图
FIGURES = FIGURE_DIR 

fig, ax = plt.subplots(figsize=(8, 4.5))

x = freq_df["freq_group"]
y_users = freq_df["用户占比%"]
y_records = freq_df["购买贡献%"]

bar_width = 0.35
x_pos = range(len(x))

bars1 = ax.bar([p - bar_width/2 for p in x_pos], y_users, bar_width, label="用户占比%", color="#5DADE2")
bars2 = ax.bar([p + bar_width/2 for p in x_pos], y_records, bar_width, label="购买贡献%", color="#E74C3C")

ax.set_xticks(x_pos)
ax.set_xticklabels(x)
ax.set_ylabel("百分比(%)")
ax.set_title("购买用户频次分层:人数占比 vs 购买贡献")
ax.legend()

# 加上数值标签
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, height, f"{height:.1f}%", ha="center", va="bottom", fontsize=10)

fig.tight_layout()
out_path = FIGURE_DIR / "buy_freq_segments.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()

print("图已保存:", out_path)

#购买频次分群核心发现

1. 业绩高度依赖高频买家:在 9 天窗口内购买 3 次及以上的“高频买家”仅占总购买人数的 42.6%,
   但他们却贡献了高达 73.1% 的购买行为。这部分用户是平台 GMV(商品交易总额)的绝对主力。

2. 一次性买家基数大但贡献低:有 34.0% 的用户在窗口内仅购买过一次,但他们只贡献了 11.3% 的购买记录。
   这说明拉新或首次转化虽然能带来大量人头,但单次购买的价值有限,不促其复购的话,对平台整体营收拉动不明显。

3. 复购买家具备极高转化潜力:购买 2 次的用户占比 23.4%,贡献了 15.6% 的购买行为。
   这部分用户已经跨过了“信任门槛”,是最容易被营销活动引导向“高频买家”转化的人群。

4. 业务建议:
   - 针对一次性买家(34%):核心痛点是“留存”,应重点通过“首单后定向发券”、“关联推荐”等手段刺激其第二次购买;
   - 针对 2 次复购买家(23.4%):提供“满减凑单”、“会员积分加倍”等活动,推高其客单价与购买频次,向核心高价值圈层跃迁;
   - 针对 3 次+ 核心买家(42.6%):需建立 VIP 专属权益(如极速退款、专享客服、新品优先体验),维护其忠诚度,防止被竞品挖走。

In [ ]:
#导出汇总表 + 关闭连接

# 把三张核心表存到 processed 目录
from src.config import PROCESSED_DIR
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

hourly_activity.to_csv(PROCESSED_DIR / "hourly_activity.csv", index=False, encoding="utf-8-sig")
funnel_table.to_csv(PROCESSED_DIR / "conversion_funnel.csv", index=False, encoding="utf-8-sig")
freq_df.to_csv(PROCESSED_DIR / "buy_freq_segments.csv", index=False, encoding="utf-8-sig")

print("汇总表已导出至:", PROCESSED_DIR)

# 关闭只读连接
con.close()
print("✅ 03 深度分析完成,只读连接已关闭。")